In [ ]:
import pynq
from pynq import Overlay
import numpy as np

print("Loading FPGA Bitstream...")
overlay = Overlay('./design_123.bit')

print("Checking available IP blocks in the overlay:")
for key in overlay.ip_dict.keys():
    print(" -", key)

try:
    dma = overlay.axi_dma_0
    print("Successfully connected to DMA!")
except AttributeError:
    print("Error: Could not find 'axi_dma_0'. Please check the exact name in the printed list above.")

print("Allocating memory for DMA transfer...")
input_buffer = pynq.allocate(shape=(784,), dtype=np.uint8)
np.copyto(input_buffer, np.random.randint(0, 255, size=(784,), dtype=np.uint8))

output_buffer = pynq.allocate(shape=(16,), dtype=np.int32)
output_buffer.fill(0)

print("Running hardware inference...")
try:
    dma.recvchannel.transfer(output_buffer)
    dma.sendchannel.transfer(input_buffer)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    print("Hardware Inference Complete!")
    print("Raw Output: ", output_buffer)
except Exception as e:
    print("DMA Transfer failed:", e)

input_buffer.freebuffer()
output_buffer.freebuffer()
